In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

data_path = '/content/drive/My Drive/multilingual-health-qa/'
print(os.listdir(data_path))

['Val.csv', 'SampleSubmission.csv', 'Test.csv', 'Train.csv']


# Preprocessing Pipeline

This notebook cleans and prepares the Zindi Multilingual Health QA dataset for model training.
We preserve special African language characters and avoid lowercasing.

In [4]:
import pandas as pd
import numpy as np
import re

print("Libraries loaded!")

Libraries loaded!


## Load Data
We load all three splits from Google Drive. Train has input/output pairs, Test has only inputs — our model must generate the outputs.

In [7]:
data_path = '/content/drive/My Drive/multilingual-health-qa/'

train = pd.read_csv(data_path + 'Train.csv')
val   = pd.read_csv(data_path + 'Val.csv')
test  = pd.read_csv(data_path + 'Test.csv')
sample_sub = pd.read_csv(data_path + 'SampleSubmission.csv')

print(f'Train : {train.shape}')
print(f'Val   : {val.shape}')
print(f'Test  : {test.shape}')

Train : (29815, 4)
Val   : (6686, 4)
Test  : (2618, 3)


## Text Cleaning Function
We clean the text by removing extra whitespace and newlines while preserving special African language characters like ɔ, ɛ, ɲ, ŋ, ɣ. We do NOT lowercase because African languages are case sensitive.

In [8]:
def clean_text(text):
    if pd.isna(text):
        return text
    # Remove extra whitespace and newlines
    text = str(text).strip()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

# Test it
sample = "  Ɔkwan bɛn so\n\n na ɲan   yɛ adwuma?  "
print("Before:", repr(sample))
print("After :", repr(clean_text(sample)))

Before: '  Ɔkwan bɛn so\n\n na ɲan   yɛ adwuma?  '
After : 'Ɔkwan bɛn so na ɲan yɛ adwuma?'


## Apply Cleaning to All Splits
We apply the clean_text function to the input and output columns across all three splits.

In [9]:
train_clean = train.copy()
val_clean   = val.copy()
test_clean  = test.copy()

train_clean['input']  = train_clean['input'].apply(clean_text)
train_clean['output'] = train_clean['output'].apply(clean_text)

val_clean['input']  = val_clean['input'].apply(clean_text)
val_clean['output'] = val_clean['output'].apply(clean_text)

test_clean['input'] = test_clean['input'].apply(clean_text)

print("Cleaning applied!")
print(f"Train: {train_clean.shape}")
print(f"Val  : {val_clean.shape}")
print(f"Test : {test_clean.shape}")

Cleaning applied!
Train: (29815, 4)
Val  : (6686, 4)
Test : (2618, 3)


## Tokenizer Analysis
We load the mT5-base tokenizer to verify special characters are preserved and to confirm our max_input_length and max_target_length settings from EDA.

In [10]:
!pip install transformers sentencepiece -q

from transformers import AutoTokenizer

MODEL_NAME = 'google/mt5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer loaded! Vocab size: 250,100


## Verify Special Characters Are Preserved
We verify that African language special characters survive tokenization and decoding — this is critical for language quality.

In [11]:
SPECIAL_CHARS = ['ɔ', 'Ɔ', 'ɛ', 'Ɛ', 'ɲ', 'ŋ', 'ɣ']

test_sentence = "Ɔkwan bɛn so na ɲan yɛ ŋ ɣ adwuma?"
token_ids = tokenizer(test_sentence, add_special_tokens=False)['input_ids']
decoded   = tokenizer.decode(token_ids)

print('Original:', test_sentence)
print('Decoded :', decoded)
print()

for ch in SPECIAL_CHARS:
    status = 'OK' if ch in decoded else 'MISSING'
    print(f"  {ch} → {status}")

Original: Ɔkwan bɛn so na ɲan yɛ ŋ ɣ adwuma?
Decoded : Ɔkwan bɛn so na ɲan yɛ ŋ ɣ adwuma?

  ɔ → MISSING
  Ɔ → OK
  ɛ → OK
  Ɛ → MISSING
  ɲ → OK
  ŋ → OK
  ɣ → OK


## Token Length Analysis
We check what percentage of inputs and outputs exceed our chosen limits of 128 and 256 tokens. This confirms our tokenization settings from EDA are appropriate.

In [12]:
# Sample 1000 examples for speed
sample = train_clean.sample(1000, random_state=42)

input_lengths  = [len(tokenizer(t)['input_ids']) for t in sample['input']]
output_lengths = [len(tokenizer(t)['input_ids']) for t in sample['output']]

input_exceed  = sum(l > 128 for l in input_lengths)
output_exceed = sum(l > 256 for l in output_lengths)

print(f'Inputs exceeding 128 tokens : {input_exceed/10:.1f}%')
print(f'Outputs exceeding 256 tokens: {output_exceed/10:.1f}%')
print()
print(f'Avg input tokens : {sum(input_lengths)/len(input_lengths):.1f}')
print(f'Avg output tokens: {sum(output_lengths)/len(output_lengths):.1f}')

Inputs exceeding 128 tokens : 0.3%
Outputs exceeding 256 tokens: 16.9%

Avg input tokens : 28.6
Avg output tokens: 149.8


## Save Cleaned Data
We save the cleaned splits back to Google Drive for use in training.

In [13]:
train_clean.to_csv(data_path + 'train_clean.csv', index=False)
val_clean.to_csv(data_path + 'val_clean.csv', index=False)
test_clean.to_csv(data_path + 'test_clean.csv', index=False)

print('Cleaned files saved to Google Drive!')
print(f'train_clean: {train_clean.shape}')
print(f'val_clean  : {val_clean.shape}')
print(f'test_clean : {test_clean.shape}')

Cleaned files saved to Google Drive!
train_clean: (29815, 4)
val_clean  : (6686, 4)
test_clean : (2618, 3)


## Preprocessing Summary

| Decision | Choice | Reason |
|---|---|---|
| Lowercasing | No | African languages are case sensitive |
| Special characters | Preserved | ɔ, ɛ, ɲ, ŋ, ɣ are valid language characters |
| max_input_length | 128 tokens | Only 0.3% of inputs exceed this |
| max_target_length | 256 tokens | Covers ~83% of outputs — will test 512 in experiments |
| Tokenizer | mT5-base | Multilingual model supporting all 8 language subsets |
| Null handling | Not needed | Dataset has zero missing values |